In [4]:
import numpy as np

# name of the file
fgrids = 'Grids_Mcdm_IllustrisTNG_1P_128_z=0.0.npy'

# read the data
grids = np.load(fgrids, mmap_mode='r')

# name of the file
fparams = 'params_1P_IllustrisTNG.txt'

# read the data
params = np.loadtxt(fparams)

grid_number = 0
params_map  = params[grid_number]
print(params_map)
# take the first 3D grid
#grids[0]



[0.30000001 0.80000001 1.         1.         1.         1.        ]


In [2]:
import h5py, numpy as np

halo_file = "groups_090_1P_0.hdf5" 

with h5py.File(halo_file, 'r') as f:
    # list group keys
    print("Group keys:", list(f.keys()))
    if 'Header' in f:
        try:
            hdr = dict(f['Header'].attrs)
            print("Header attrs:", hdr)
        except Exception as e:
            print("Header read error:", e)
            
    # check Group keys and some quantities
    print("Group keys:", list(f['Group'].keys()))
    
    group_pos = np.array(f['Group/GroupPos'])
    group_vel = np.array(f['Group/GroupVel'])

    
    #print("group_pos shape, sample (ckpc/h):", group_pos.shape, group_pos[:5])
    #print("group_vel shape, sample:", group_vel.shape, group_vel[:5])
    # stats on velocities (componentwise and magnitude)
    vx = group_vel[:,0]; vy = group_vel[:,1]; vz = group_vel[:,2]
    #vmag = np.sqrt(vx**2 + vy**2 + vz**2)
    #print("vx stats: mean,std,min,max", vx.mean(), vx.std(), vx.min(), vx.max())
    #print("vz stats: mean,std,min,max", vz.mean(), vz.std(), vz.min(), vz.max())
    #print("|v| stats: mean,std,min,max", vmag.mean(), vmag.std(), vmag.min(), vmag.max())
    


Group keys: ['Config', 'Group', 'Header', 'IDs', 'Parameters', 'Subhalo']
Header attrs: {'BoxSize': np.float64(25000.0), 'FlagDoubleprecision': np.int32(0), 'Git_commit': np.bytes_(b'4ab97a2c5659df3e83e01c53f8142816fc0db675'), 'Git_date': np.bytes_(b'Tue May 3 10:27:55 2016 +0200'), 'HubbleParam': np.float64(0.6711), 'Ngroups_ThisFile': np.int32(20817), 'Ngroups_Total': np.int32(20817), 'Nids_ThisFile': np.int32(12770445), 'Nids_Total': np.int64(12770445), 'Nsubgroups_ThisFile': np.int32(18635), 'Nsubgroups_Total': np.int32(18635), 'NumFiles': np.int32(1), 'Omega0': np.float64(0.3), 'OmegaLambda': np.float64(0.7), 'Redshift': np.float64(2.220446049250313e-16), 'Time': np.float64(0.9999999999999998)}
Group keys: ['GroupBHMass', 'GroupBHMdot', 'GroupCM', 'GroupFirstSub', 'GroupGasMetalFractions', 'GroupGasMetallicity', 'GroupLen', 'GroupLenType', 'GroupMass', 'GroupMassType', 'GroupNsubs', 'GroupPos', 'GroupSFR', 'GroupStarMetalFractions', 'GroupStarMetallicity', 'GroupVel', 'GroupWindMa

In [2]:
import h5py
import numpy as np

halo_file = "groups_090_1P_0.hdf5"

with h5py.File(halo_file, "r") as f:

    
    if "Group/Group_M_Mean200" in f:
        mass_mean200 = np.array(f["Group/Group_M_Mean200"])* 1e10
        print("Group_M_Mean200:")
        print("  shape:", mass_mean200.shape)
        print("  min, max, mean =",
              np.min(mass_mean200),
              np.max(mass_mean200),
              np.mean(mass_mean200))
    else:
        print("Group_M_Mean200 not found")

   
    if "Group/GroupMass" in f:
        mass_fof = np.array(f["Group/GroupMass"])* 1e10
        print("\nGroupMass (FoF):")
        print("  shape:", mass_fof.shape)
        print("  min, max, mean =",
              np.min(mass_fof),
              np.max(mass_fof),
              np.mean(mass_fof))
    else:
        print("GroupMass not found")

    if "Group/GroupNsubs" in f:
        groupNsubs = np.array(f["Group/GroupNsubs"])
        print("GroupNsubs:",groupNsubs)
    else:
        print("GroupNsubs not found")


    if "Subhalo/SubhaloMass" in f:
        sub_mass = np.array(f["Subhalo/SubhaloMass"]) * 1e10
        print("\nSubhaloMass:")
        print("  shape:", sub_mass.shape)
        print("  min subhalo_mass, max subhalo_mass =",
              np.min(sub_mass),
              np.max(sub_mass))
              #np.mean(sub_mass))
    else:
        print("GroupMass not found")

    subhalo_mass_type = f['Subhalo/SubhaloMassType'][:]  
    subhalo_group = f['Subhalo/SubhaloGrNr'][:]           

    # stellar mass (type 4)
    M_star_sub = subhalo_mass_type[:, 4] * 1e10  # Msun/h
    print("max M_star_sub=", np.max(M_star_sub))
    print("min M_star_sub=", np.min(M_star_sub))

    M_DM = f['Subhalo/SubhaloMassType'][:,1]*1e10
    # number of groups
    Ngroups = f['Group/GroupPos'].shape[0]

    # initialize group stellar mass
    M_star_group = np.zeros(Ngroups)

    # sum stellar mass into each group
    for i in range(len(M_star_sub)):
        g = subhalo_group[i]
        M_star_group[g] += M_star_sub[i]

    print("Total stellar mass (subhalos):", np.sum(M_star_sub))
    print("Total stellar mass (groups):", np.sum(M_star_group))
    print("min DM mass=", np.min(M_DM))
    print("max DM mass=", np.max(M_DM))

Group_M_Mean200:
  shape: (20817,)
  min, max, mean = 0.0 1.0473231e+14 2.8906551e+10

GroupMass (FoF):
  shape: (20817,)
  min, max, mean = 3.9444346e+08 9.69151e+13 2.8474591e+10
GroupNsubs: [505  88  73 ...   1   1   0]

SubhaloMass:
  shape: (18635,)
  min subhalo_mass, max subhalo_mass = 2.6162238e+08 8.798259e+13
max M_star_sub= 5.0591934e+11
min M_star_sub= 0.0
Total stellar mass (subhalos): 5.389141e+12
Total stellar mass (groups): 5389140914736.5
min DM mass= 0.0
max DM mass= 7.553909e+13
